# HeH⁺ potential energy surface: from exact references to real quantum hardware
**Team:** *Name A* (algorithm & physics), *Name B* (hardware & noise)

**BasQ Qiskit Fall Fest 2026, Challenge 1, Beginner track**

## Abstract
*TODO (both, last 10 min): 4–5 sentences. System, what we pushed beyond the baseline, main numbers (equilibrium length, D_e, best hardware error vs exact), and the main limitation.*

## How this notebook scores against the judging criteria
| Criterion | Where we address it |
|---|---|
| **Technical complexity** | Section 1 qubit tapering 4→2 qubits; Section 4 larger basis (6-31G), ansatz and optimizer study |
| **Validation** | Section 2 independent PySCF FCI cross-check; Section 3 VQE vs exact over the full curve; Section 6 noisy simulation; Section 7 real hardware; Section 8 all levels compared |
| **Hardware awareness** | Section 5 encoding vs routing depth, noise-aware layout selection; Section 6 ZNE; Section 7 dynamical decoupling, twirling, TREX on a real device |
| **Performance and limitations** | Section 8 error budget (ansatz / shots / noise / mitigation residual) and scaling |
| **Presentation** | Reproducibility below; each section states its claim, evidence and caveat |

## Reproducibility
```bash
pip install -r requirements.txt          # pinned versions
python beginner/check_setup.py           # must print OK
```
Run top to bottom (≈ *TODO* min without hardware). Hardware results are loaded from `results/hardware_*.csv` unless `RUN_HARDWARE = True`. All energies are **total energies in Hartree**. Chemical accuracy = 1.6 mHa.

## Team plan (delete this cell before submitting)

| Time | A: algorithm & physics | B: hardware & noise |
|---|---|---|
| 0:00–0:10 | Read the template together, run Section 0, agree on the distance grid | same |
| 0:10–0:50 | Sections 1, 2, 3 (noiseless PES, equilibrium, dissociation) | Section 5 transpile study, then **submit the hardware job by ~0:45** (Section 7) because of queue time |
| 0:50–1:25 | Section 4 beyond the baseline (6-31G, ansatz/optimizer) | Section 6 noisy PES and ZNE; collect hardware results when they're ready |
| 1:25–1:45 | Section 8 error budget and combined figure (A writes it, B supplies numbers) | Section 7 hardware analysis, Section 8 noise rows |
| 1:45–2:00 | Abstract and conclusions, **Restart & Run All**, export HTML | Proofread, check figures and job IDs, final commit |

**Git rules for the next two hours:** only edit cells in your own sections. Experiment in `scratch_<name>.ipynb`. Every section saves `results/*.csv` and `figures/*.png`. Pull before you start, and commit and push every ~15 min.

# 0. Setup

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from pes_tools import (CHEMICAL_ACCURACY, HARTREE_TO_KCAL_MOL, build_problem,
                       hf_energy, exact_energy, vqe_energy, uccsd_ansatz)
from hardware_tools import (make_mapper, prepare_point, transpile_report, best_transpile,
                            noisy_estimator, estimate, fold_two_qubit_gates, zne_extrapolate,
                            runtime_estimator)

warnings.filterwarnings("ignore", module="scipy")
RESULTS, FIGURES = Path("results"), Path("figures")
RUN_HARDWARE = False  # flip to True only to (re)submit real-device jobs

# Shared distance grid (Angstrom). Agree on this in the first 10 minutes.
DISTANCES = np.round(np.concatenate([np.linspace(0.5, 1.2, 8), np.linspace(1.4, 3.5, 6)]), 3)  # TODO
HW_DISTANCES = [0.6, 0.774, 1.0, 1.5, 2.5]  # TODO: subset for noisy sim / hardware

# 1. System and qubit encoding
- **Owner:** A
- **Scores on:** Technical complexity, Presentation
- **Timebox:** 0:10–0:20

HeH⁺: 2 electrons (He 2 + H 1 − 1 for the charge), closed-shell singlet. In STO-3G that is 2 spatial orbitals, i.e. 4 spin-orbitals.

**Claim:** exploiting particle-number and ℤ₂ symmetries shrinks the problem from 4 qubits (Jordan-Wigner) to 2 (parity + tapering) with **no loss of accuracy**, and that is what makes the hardware runs in Sections 5–7 feasible.

*TODO: one short paragraph on why tapering is exact here (symmetry sectors), and a reference.*

In [ ]:
problem = build_problem(0.774)
rows = []
for kind in ["jw", "parity", "tapered"]:
    m = make_mapper(problem, kind)
    op = m.map(problem.hamiltonian.second_q_op())
    a = uccsd_ansatz(problem, m)
    rows.append({"encoding": kind, "qubits": op.num_qubits, "pauli_terms": len(op),
                 "uccsd_params": a.num_parameters, "exact_Ha": exact_energy(problem, m)})
encodings = pd.DataFrame(rows); encodings

*TODO (A): 2–3 sentences interpreting the table.*

# 2. Classical references and an independent check
- **Owner:** A
- **Scores on:** Validation
- **Timebox:** 0:20–0:25

Before trusting any quantum number, we check our "exact" reference two independent ways: Qiskit Nature's diagonalisation and PySCF's own FCI solver.

**Pitfall we found:** the tutorial's unfiltered `NumPyMinimumEigensolver` searches every electron count. For HeH⁺ it returns a state with extra electrons, ≈0.16 Ha too low. We filter on particle number and spin (`exact_energy` in `pes_tools.py`).

In [ ]:
from pyscf import gto, scf, fci
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

def pyscf_fci(d, basis="sto3g"):
    mol = gto.M(atom=f"He 0 0 0; H 0 0 {d}", basis=basis, charge=1, spin=0, verbose=0)
    mf = scf.RHF(mol).run()
    return mf.e_tot, fci.FCI(mf).kernel()[0]

e_rhf, e_fci = pyscf_fci(0.774)
p = build_problem(0.774)
unfiltered = GroundStateEigensolver(make_mapper(p, "jw"), NumPyMinimumEigensolver()).solve(p).total_energies[0]
print(f"PySCF RHF          {e_rhf: .8f}   Qiskit HF circuit {hf_energy(p): .8f}")
print(f"PySCF FCI          {e_fci: .8f}   Qiskit exact      {exact_energy(p): .8f}")
print(f"Unfiltered (wrong) {unfiltered: .8f}   off by {unfiltered - e_fci:+.3f} Ha")

# 3. Noiseless potential energy surface
- **Owner:** A
- **Scores on:** Validation, Performance
- **Timebox:** 0:25–0:50

Noiseless VQE (UCCSD, tapered 2 qubits, SLSQP) against HF and exact over the whole curve. Each point is warm-started from the previous one's parameters, and Section 8 compares that with cold starts.

In [ ]:
rows, prev = [], None
for d in DISTANCES:
    prob = build_problem(float(d)); m = make_mapper(prob, "tapered")
    v = vqe_energy(prob, mapper=m, initial_point=prev)
    prev = v.optimal_point
    rows.append({"distance_A": d, "hf": hf_energy(prob, m), "vqe": v.energy,
                 "exact": exact_energy(prob, m), "evals": v.cost_function_evals})
pes = pd.DataFrame(rows)
pes["vqe_err"] = (pes.vqe - pes.exact).abs()
pes["correlation"] = pes.exact - pes.hf
pes.to_csv(RESULTS / "pes_noiseless_sto3g.csv", index=False)
pes

In [ ]:
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, height_ratios=[3, 1])
ax.plot(pes.distance_A, pes.hf, "o--", c="#9e9e9e", label="Hartree-Fock")
ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(pes.distance_A, pes.vqe, "x", c="#1f77b4", ms=9, label="VQE (UCCSD, 2 qubits)")
ax.set_ylabel("Total energy (Ha)"); ax.legend(); ax.set_title("HeH$^+$ PES, STO-3G, noiseless")
ax2.semilogy(pes.distance_A, pes.vqe_err, "o-", c="#1f77b4", label="|VQE − exact|")
ax2.axhline(CHEMICAL_ACCURACY, c="red", ls=":", label="chemical accuracy")
ax2.set_xlabel("He–H distance (Å)"); ax2.set_ylabel("Error (Ha)"); ax2.legend()
fig.tight_layout(); fig.savefig(FIGURES / "pes_noiseless_sto3g.png", dpi=150)

### 3a. Equilibrium bond length

In [ ]:
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar
spl = CubicSpline(pes.distance_A, pes.vqe)
opt = minimize_scalar(spl, bounds=(pes.distance_A.min(), pes.distance_A.max()), method="bounded")
r_eq, e_eq = opt.x, float(opt.fun)
print(f"r_eq (VQE, spline) = {r_eq:.4f} Å,  E = {e_eq:.6f} Ha")
# TODO (A): refine with VQE directly at r_eq; compare to the experimental value (cite it), and comment on basis-set error.

### 3b. Dissociation

In [ ]:
def atom_fci(symbol, charge, spin, basis="sto3g"):
    nelec = {"He": 2, "H": 1}[symbol] - charge
    if nelec == 0:
        return 0.0  # a bare proton has no electrons
    mol = gto.M(atom=f"{symbol} 0 0 0", basis=basis, charge=charge, spin=spin, verbose=0)
    mf = (scf.RHF if spin == 0 else scf.ROHF)(mol).run()
    return fci.FCI(mf).kernel()[0]

channels = {"He + H+": atom_fci("He", 0, 0) + atom_fci("H", 1, 0),
            "He+ + H": atom_fci("He", 1, 1) + atom_fci("H", 0, 1)}
for k, v in channels.items(): print(f"{k:8s} {v: .6f} Ha")
e_inf = min(channels.values())
print(f"Long-distance end of our curve: {pes.exact.iloc[-1]:.6f} Ha")
print(f"D_e ≈ {(e_inf - e_eq):.4f} Ha = {(e_inf - e_eq) * HARTREE_TO_KCAL_MOL:.1f} kcal/mol")
# TODO (A): which channel wins and why (ionisation energies of He vs H)? Is the curve converged at 3.5 Å?

*TODO (A): interpret Section 3. Where does correlation energy matter most along the curve? Does VQE stay within chemical accuracy everywhere?*

# 4. Beyond the baseline
- **Owner:** A
- **Scores on:** Technical complexity, Performance
- **Timebox:** 0:50–1:25

Pick the ones that time allows, in this order:
1. **Larger basis (6-31G):** 4 spatial orbitals, 6 tapered qubits, 15 UCCSD parameters. Does basis-set improvement outweigh the ansatz error? Sweep a few distances.
2. **Ansatz comparison:** UCCSD vs a hardware-efficient `efficient_su2` (fewer 2q gates, but needs many more parameters and can get stuck in local minima).
3. **Optimizer comparison:** SLSQP vs COBYLA vs SPSA. SPSA is the realistic choice under shot noise.

In [ ]:
rows = []
for basis in ["sto3g", "6-31g"]:
    for kind in ["jw", "tapered"]:
        t0 = time.perf_counter()
        prob = build_problem(0.774, basis=basis); m = make_mapper(prob, kind)
        v = vqe_energy(prob, mapper=m)
        rows.append({"basis": basis, "encoding": kind, "qubits": v.num_qubits, "params": v.num_parameters,
                     "vqe": v.energy, "exact": exact_energy(prob, m), "seconds": time.perf_counter() - t0})
scaling = pd.DataFrame(rows); scaling["err"] = (scaling.vqe - scaling.exact).abs()
scaling.to_csv(RESULTS / "basis_encoding_scaling.csv", index=False); scaling

In [ ]:
# TODO (A): 6-31G PES on a coarse grid; ansatz/optimizer comparison. Save to results/.

*TODO (A): interpretation.*

# 5. Hardware-aware compilation
- **Owner:** B
- **Scores on:** Hardware awareness
- **Timebox:** 0:10–0:35

**Claim:** the choice of encoding sets the circuit size on a real chip, and circuit size sets the noise. We compare encodings and transpiler optimisation levels, then pick the physical qubits with the lowest total reported error (gate + readout errors from the device's calibration data), scanning 20 transpiler seeds. A deliberately bad layout shows the choice matters.

Backend: `FakeFez`, a snapshot of the calibration data of IBM's 156-qubit Heron r2 device `ibm_fez`. If Section 7 runs on another device, swap in its fake copy (e.g. `FakeMarrakesh`, `FakeTorino`) so the noise model matches.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeFez
backend = FakeFez()

rows = []
for kind in ["jw", "tapered"]:
    pp = prepare_point(0.774, mapping=kind)
    for lvl in [0, 1, 2, 3]:
        r = transpile_report(pp.circuit, backend, optimization_level=lvl, seed=0)
        rows.append({"encoding": kind, "opt_level": lvl, "depth": r["depth"],
                     "two_qubit_gates": r["two_qubit_gates"], "error_score": r["error_score"]})
compile_study = pd.DataFrame(rows)
compile_study.to_csv(RESULTS / "compile_study.csv", index=False); compile_study

In [ ]:
pp_eq = prepare_point(0.774, mapping="tapered")
best = best_transpile(pp_eq.circuit, backend, n_seeds=20)
print("chosen physical qubits:", best["physical_qubits"], "| depth", best["depth"],
      "| 2q gates", best["two_qubit_gates"], "| error score", round(best["error_score"], 4))

# Deliberately BAD layout: the connected qubit pair with the worst CZ + readout errors.
t = backend.target
def pair_cost(a, b):
    return t["cz"][(a, b)].error + t["measure"][(a,)].error + t["measure"][(b,)].error
pairs = [q for q, p in t["cz"].items() if p is not None and p.error is not None and p.error < 1]  # skip broken links
worst_pair = max(pairs, key=lambda q: pair_cost(*q))
worst = transpile_report(pp_eq.circuit, backend, initial_layout=list(worst_pair))

est = noisy_estimator(backend)
rows = []
for name, r in [("noise-aware (best)", best), ("worst qubits", worst)]:
    ev, _ = estimate([r["isa"]], pp_eq.qubit_op, est, precision=0)  # precision=0: exact noisy average
    rows.append({"layout": name, "physical_qubits": r["physical_qubits"], "two_qubit_gates": r["two_qubit_gates"],
                 "error_score": r["error_score"],
                 "energy_error_mHa": 1e3 * (ev[0] + pp_eq.nuclear_repulsion - pp_eq.e_exact)})
layout_study = pd.DataFrame(rows)
layout_study.to_csv(RESULTS / "layout_study.csv", index=False)
display(layout_study)
best["isa"].draw("mpl", idle_wires=False, fold=-1)

**What this shows.**
- **Encoding matters most.** After compiling for the chip, the 4-qubit Jordan-Wigner circuit needs **45 CZ gates (depth 140)**. The 2-qubit tapered circuit needs **2 CZ gates (depth 11)**: 22× fewer two-qubit gates for the same exact answer.
- **Optimisation level matters too.** Levels 0–1 keep 4 CZs and place the circuit on qubits 0–1 without looking at their quality. Levels 2–3 simplify to 2 CZs *and* choose better qubits (error score 0.070 → 0.019).
- **Qubit choice matters.** The noise-aware layout picked qubits 22–23. On the worst connected pair (113–119, readout error ≈ 11 %), the same circuit has **17.6 mHa** error instead of **7.7 mHa**: 2.3× worse from where it runs alone.

# 6. Noisy simulation and error mitigation
- **Owner:** B
- **Scores on:** Validation, Hardware awareness
- **Timebox:** 0:50–1:25

We optimise parameters noiselessly (Section 3), then **evaluate** them under the backend's noise model. That separates noise error from optimisation error.

We use `precision=0`: Aer then returns the *exact average* energy under the noise model, with no shot noise, so what we see is purely the bias caused by noise. Shot noise is measured for real in Section 7.

ZNE: fold each 2-qubit gate (scale factors 1, 3, 5, i.e. 1×, 3×, 5× the two-qubit noise) and extrapolate back to zero noise with a straight line (linear) and a parabola (quadratic). Error bars on ZNE values are propagated with `zne_std`.

In [ ]:
from hardware_tools import zne_std
est = noisy_estimator(backend)
SCALES = [1, 3, 5]
SIM_PRECISION = 0  # 0 = exact noisy average (no shot noise)
rows = []
for d in HW_DISTANCES:
    pp = prepare_point(d, mapping="tapered")
    isa = best_transpile(pp.circuit, backend, n_seeds=10)["isa"]
    ev, sd = estimate([fold_two_qubit_gates(isa, s) for s in SCALES], pp.qubit_op, est, precision=SIM_PRECISION)
    E = ev + pp.nuclear_repulsion
    rows.append({"distance_A": d, "exact": pp.e_exact, "vqe_noiseless": pp.e_vqe,
                 "noisy": E[0], "noisy_std": sd[0],
                 "zne_linear": zne_extrapolate(SCALES, E, 1), "zne_linear_std": zne_std(SCALES, sd, 1),
                 "zne_quad": zne_extrapolate(SCALES, E, 2), "zne_quad_std": zne_std(SCALES, sd, 2),
                 **{f"scale_{s}": e for s, e in zip(SCALES, E)}})
noisy = pd.DataFrame(rows)
for c in ["noisy", "zne_linear", "zne_quad"]:
    noisy[f"{c}_err"] = (noisy[c] - noisy.exact).abs()
noisy.to_csv(RESULTS / "pes_noisy_sim.csv", index=False)
noisy[["distance_A", "exact", "noisy", "zne_linear", "zne_quad", "noisy_err", "zne_linear_err", "zne_quad_err"]]

In [ ]:
fig, (ax, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))
if "pes" in globals():  # continuous exact curve from Section 3
    ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(noisy.distance_A, noisy.noisy, "o", c="#d62728", label="Noisy sim, raw")
ax.plot(noisy.distance_A, noisy.zne_linear, "s", c="#1f77b4", label="Noisy sim + ZNE (linear)")
ax.plot(noisy.distance_A, noisy.zne_quad, "^", c="#9467bd", mfc="none", label="Noisy sim + ZNE (quadratic)")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)"); ax.set_title("Noisy simulation (FakeFez)"); ax.legend()

for c, m, col in [("noisy", "o", "#d62728"), ("zne_linear", "s", "#1f77b4"), ("zne_quad", "^", "#9467bd")]:
    ax2.semilogy(noisy.distance_A, noisy[f"{c}_err"] * 1e3, m + "-", c=col, label=c)
ax2.axhline(CHEMICAL_ACCURACY * 1e3, c="k", ls=":", label="chemical accuracy")
ax2.set_xlabel("He–H distance (Å)"); ax2.set_ylabel("|E − exact| (mHa)"); ax2.set_title("Error vs exact"); ax2.legend()

row = noisy.loc[noisy.distance_A.sub(0.774).abs().idxmin()]
E = [row[f"scale_{s}"] for s in SCALES]; x = np.linspace(0, max(SCALES), 50)
ax3.plot(SCALES, E, "o", c="#d62728", ms=8, label="measured (scales 1, 3, 5)")
for order, col in [(1, "#1f77b4"), (2, "#9467bd")]:
    ax3.plot(x, np.polyval(np.polyfit(SCALES, E, order), x), "--", c=col, label=f"{'linear' if order == 1 else 'quadratic'} fit")
ax3.axhline(row.exact, c="#2ca02c", label="exact")
ax3.set_xlabel("noise scale factor"); ax3.set_ylabel("Total energy (Ha)"); ax3.set_title(f"ZNE at {row.distance_A} Å"); ax3.legend()
fig.tight_layout(); fig.savefig(FIGURES / "noisy_sim_zne.png", dpi=150)

from hardware_tools import two_qubit_count
# Extra: same pipeline for the 4-qubit Jordan-Wigner circuit at equilibrium (deeper circuit -> more noise?)
rows = []
for kind in ["tapered", "jw"]:
    p = prepare_point(0.774, mapping=kind)
    isa = best_transpile(p.circuit, backend, n_seeds=10)["isa"]
    ev, _ = estimate([fold_two_qubit_gates(isa, s) for s in SCALES], p.qubit_op, est, precision=0)
    Ek = ev + p.nuclear_repulsion
    rows.append({"encoding": kind, "two_qubit_gates": two_qubit_count(isa),
                 "raw_err_mHa": 1e3 * abs(Ek[0] - p.e_exact),
                 "zne_linear_err_mHa": 1e3 * abs(zne_extrapolate(SCALES, Ek, 1) - p.e_exact)})
encoding_noise = pd.DataFrame(rows); encoding_noise.to_csv(RESULTS / "encoding_noise.csv", index=False)
encoding_noise

**What this shows.**
- **Raw noise error is 5–10 mHa** (3–6× chemical accuracy) even for a circuit with only 2 CZ gates. It is largest at short distances.
- **ZNE removes about 70–75 % of it**, leaving **1.3–2.6 mHa**. That is within chemical accuracy (1.6 mHa) at 1.5 Å and 2.5 Å, and just outside it at 0.6–1.0 Å.
- **Linear and quadratic agree to within 0.1 mHa.** The energy rises almost exactly in a straight line with the noise scale (right panel), so the simpler linear fit is enough. On hardware the linear fit is also the better choice because it inflates shot noise less (error bar ×1.21 vs ×2.28).
- **Why ZNE can't remove everything:** folding only amplifies two-qubit gate noise. Readout errors and single-qubit errors are not scaled, so they are not extrapolated away. We think that is most of the residual. On hardware, TREX corrects readout separately.
- **Deeper circuits are hopeless at this noise level.** At equilibrium, the 45-CZ Jordan-Wigner circuit is off by **121 mHa raw and 43 mHa after ZNE**, against 7.7 and 2.1 mHa for the tapered one. This is why the qubit reduction in Section 1 is essential.

# 7. Real quantum hardware
- **Owner:** B
- **Scores on:** Validation, Hardware awareness
- **Timebox:** submit by ~0:45, analyse 1:25–1:45

**Submit early:** queue times are unpredictable. We send **all distances × all ZNE scales as one job** (9 circuits), so we wait in the queue only once. The job ID is saved to `results/hardware_job.json` right away, so a crashed notebook doesn't lose the job, and a separate cell collects the results later.

**Shots:** with 4096 shots one energy wobbles by ≈13 mHa (8× chemical accuracy). We use 32768 shots (≈4.6 mHa) on 3 distances to keep the QPU time small.

Error suppression and mitigation used: dynamical decoupling (XpXm), Pauli twirling on gates and measurements, TREX readout mitigation (`resilience_level=1`), plus our own folding-based ZNE on top.

**Dry run first:** `DRY_RUN = True` sends the identical job to the fake device locally, to check the code before spending real QPU time.

Set your IBM Quantum account up once in a terminal (keep the token out of the notebook):
```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", instance="...")
```

In [ ]:
import json
from hardware_tools import build_hardware_pubs, hardware_table
HW_FILE, JOB_FILE = RESULTS / "hardware_pes.csv", RESULTS / "hardware_job.json"
HW_BACKEND_NAME = None                 # e.g. "ibm_fez"; None = least busy device you can use
HW_REAL_DISTANCES = [0.774, 1.5, 2.5]  # 3 distances x 3 scales = 9 circuits in ONE job
HW_SHOTS = 32768                       # ~4.6 mHa per energy (4096 shots would be ~13 mHa)
DRY_RUN = True                         # run the same job locally on the fake device first

if DRY_RUN:
    pubs, meta = build_hardware_pubs(HW_REAL_DISTANCES, SCALES, backend)
    res = runtime_estimator(backend, shots=4096).run(pubs).result()  # fewer shots: this only checks the code
    dry = hardware_table(res, meta, SCALES, backend.name, "local dry run")
    display(dry[["distance_A", "exact", "hw_raw", "hw_raw_std", "hw_zne_linear", "hw_zne_linear_std"]])

if RUN_HARDWARE and not JOB_FILE.exists():  # never submits twice
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    hw_backend = (service.backend(HW_BACKEND_NAME) if HW_BACKEND_NAME
                  else service.least_busy(operational=True, simulator=False, min_num_qubits=5))
    pubs, meta = build_hardware_pubs(HW_REAL_DISTANCES, SCALES, hw_backend)
    job = runtime_estimator(hw_backend, shots=HW_SHOTS).run(pubs)
    JOB_FILE.write_text(json.dumps({"job_id": job.job_id(), "backend": hw_backend.name, "shots": HW_SHOTS,
                                    "scales": SCALES, "submitted": time.strftime("%Y-%m-%d %H:%M"),
                                    "meta": meta}, indent=2))
    print("Submitted", job.job_id(), "on", hw_backend.name, "-> commit results/hardware_job.json now")

In [ ]:
# Collect the hardware results (safe to re-run: it only checks the job status until it is DONE).
if JOB_FILE.exists() and not HW_FILE.exists():
    from qiskit_ibm_runtime import QiskitRuntimeService
    info = json.loads(JOB_FILE.read_text())
    job = QiskitRuntimeService().job(info["job_id"])
    status = str(job.status())
    print(info["job_id"], "on", info["backend"], "->", status)
    if "DONE" in status:
        hardware_table(job.result(), info["meta"], info["scales"], info["backend"], info["job_id"]).to_csv(HW_FILE, index=False)

hardware = pd.read_csv(HW_FILE) if HW_FILE.exists() else None
hardware if hardware is not None else print("No hardware results yet.")

In [ ]:
if hardware is None:
    print("No hardware results yet: run the two cells above with RUN_HARDWARE = True.")
else:
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    if "pes" in globals():
        ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
    ax.plot(noisy.distance_A, noisy.noisy, "o", c="#d62728", mfc="none", label="Noisy sim, raw")
    ax.plot(noisy.distance_A, noisy.zne_linear, "s", c="#1f77b4", mfc="none", label="Noisy sim + ZNE")
    ax.errorbar(hardware.distance_A, hardware.hw_raw, yerr=hardware.hw_raw_std, fmt="o", c="#d62728", capsize=3,
                label=f"Hardware raw ({hardware.backend[0]})")
    ax.errorbar(hardware.distance_A, hardware.hw_zne_linear, yerr=hardware.hw_zne_linear_std, fmt="s", c="#1f77b4",
                capsize=3, label="Hardware + ZNE (linear)")
    ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)"); ax.set_title("Real hardware"); ax.legend()
    for c, std, m, col in [("hw_raw", "hw_raw_std", "o", "#d62728"), ("hw_zne_linear", "hw_zne_linear_std", "s", "#1f77b4")]:
        ax2.errorbar(hardware.distance_A, 1e3 * (hardware[c] - hardware.exact), yerr=1e3 * hardware[std],
                     fmt=m, c=col, capsize=3, label=c)
    ax2.axhspan(-CHEMICAL_ACCURACY * 1e3, CHEMICAL_ACCURACY * 1e3, color="k", alpha=0.1, label="chemical accuracy")
    ax2.axhline(0, c="k", lw=0.8)
    ax2.set_xlabel("He–H distance (Å)"); ax2.set_ylabel("E − exact (mHa)"); ax2.set_title("Hardware error"); ax2.legend()
    fig.tight_layout(); fig.savefig(FIGURES / "hardware_pes.png", dpi=150)

*Fill in once `hardware_pes.csv` exists. Check, in this order:*
1. *Are the hardware ZNE points within their error bars of the exact curve? (Error bars ≈ ±4.6 mHa raw and ±5.6 mHa after ZNE at 32768 shots.)*
2. *Does the noisy simulator predict the raw hardware error (sim: ≈ 5–8 mHa)? If hardware is better, credit twirling, DD and TREX, which the simulator does not include.*
3. *Did ZNE move the energies toward exact? Quote the numbers.*
4. *Main limit: shot noise. One hardware energy wobbles by more than chemical accuracy, so chemical accuracy cannot be claimed from hardware at this budget, only from the simulator.*

Local dry run on FakeFez (4096 shots, no twirling/TREX): the code works end-to-end, with raw errors of 9–15 mHa and error bars of 5–13 mHa, consistent with the noisy simulation.

# 8. Performance and limitations
- **Owner:** A (writes) + B (numbers)
- **Scores on:** Performance and limitations
- **Timebox:** 1:25–1:45

### Error budget at equilibrium (0.774 Å)
Computed by the cell below from `results/*.csv` (saved as `results/error_budget.csv`).

| Source | Simulator (Ha) | Hardware (Ha) | Evidence |
|---|---|---|---|
| Basis set (6-31G − STO-3G exact energy) | −8.1 × 10⁻² | – | Section 4 |
| Ansatz / optimiser (VQE − exact, noiseless) | 5 × 10⁻⁸ | – | Section 3 |
| Shot noise (std of one energy, 32768 shots) | 0 (exact average) | ≈ 4.6 × 10⁻³ | Section 7 |
| Gate + readout noise (raw − noiseless VQE) | +7.7 × 10⁻³ | *after hardware run* | Sections 6 / 7 |
| Residual after ZNE (linear ZNE − exact) | +2.1 × 10⁻³ | *after hardware run* | Sections 6 / 7 |

**Reading the table:** the algorithm (UCCSD + VQE) is essentially perfect. The quantum errors (noise ≈ 8 mHa, shot noise ≈ 5 mHa) are much bigger than the algorithm error. But the **basis set error (81 mHa) is ten times bigger than all of them together**: the small STO-3G basis, not the quantum computer, limits how close we are to the real molecule.

### Combined figure
The headline figure below shows every level on one plot: exact, noiseless VQE, noisy simulation (raw and ZNE) and, once available, hardware with error bars.

In [ ]:
# ---- Error budget at equilibrium (0.774 A), all numbers read from results/*.csv
eq = pd.read_csv(RESULTS / "pes_noisy_sim.csv").set_index("distance_A").loc[0.774]
sc = pd.read_csv(RESULTS / "basis_encoding_scaling.csv") if (RESULTS / "basis_encoding_scaling.csv").exists() else None
hw = pd.read_csv(RESULTS / "hardware_pes.csv").set_index("distance_A").loc[0.774] if (RESULTS / "hardware_pes.csv").exists() else None
basis_err = (sc.query("encoding == 'tapered'").set_index("basis").exact.diff().iloc[-1]) if sc is not None else np.nan
if hw is not None:
    shot = hw.hw_raw_std
else:  # no hardware yet: dry-run std (4096 shots) rescaled to the shots we plan to use
    shot = dry.hw_raw_std[dry.distance_A == 0.774].iloc[0] * np.sqrt(4096 / HW_SHOTS) if "dry" in globals() else np.nan
budget = pd.DataFrame([
    {"source": "Basis set (6-31G − STO-3G exact)", "simulator_Ha": basis_err, "hardware_Ha": np.nan, "evidence": "Section 4"},
    {"source": "Ansatz / optimiser (VQE − exact, noiseless)", "simulator_Ha": eq.vqe_noiseless - eq.exact, "hardware_Ha": np.nan, "evidence": "Section 3"},
    {"source": f"Shot noise (std of one energy, {HW_SHOTS} shots)", "simulator_Ha": 0.0, "hardware_Ha": shot, "evidence": "Section 7"},
    {"source": "Gate + readout noise (raw − noiseless VQE)", "simulator_Ha": eq.noisy - eq.vqe_noiseless,
     "hardware_Ha": hw.hw_raw - hw.vqe_noiseless if hw is not None else np.nan, "evidence": "Sections 6 / 7"},
    {"source": "Residual after ZNE (ZNE linear − exact)", "simulator_Ha": eq.zne_linear - eq.exact,
     "hardware_Ha": hw.hw_zne_linear - hw.exact if hw is not None else np.nan, "evidence": "Sections 6 / 7"},
])
budget.to_csv(RESULTS / "error_budget.csv", index=False)
display(budget.style.format({"simulator_Ha": "{:+.2e}", "hardware_Ha": "{:+.2e}"}, na_rep="–"))

# ---- Headline figure: every level of the study on one plot
pes_ = pd.read_csv(RESULTS / "pes_noiseless_sto3g.csv")
ns = pd.read_csv(RESULTS / "pes_noisy_sim.csv")
hwf = pd.read_csv(RESULTS / "hardware_pes.csv") if (RESULTS / "hardware_pes.csv").exists() else None
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(pes_.distance_A, pes_.hf, "--", c="#9e9e9e", label="Hartree-Fock")
ax.plot(pes_.distance_A, pes_.exact, "-", c="#2ca02c", lw=2, label="Exact (FCI)")
ax.plot(pes_.distance_A, pes_.vqe, "x", c="k", ms=8, label="VQE noiseless")
ax.plot(ns.distance_A, ns.noisy, "o", c="#d62728", mfc="none", ms=8, label="Noisy sim, raw")
ax.plot(ns.distance_A, ns.zne_linear, "s", c="#1f77b4", mfc="none", ms=8, label="Noisy sim + ZNE")
if hwf is not None:
    ax.errorbar(hwf.distance_A, hwf.hw_raw, yerr=hwf.hw_raw_std, fmt="o", c="#d62728", capsize=3, label=f"Hardware raw ({hwf.backend[0]})")
    ax.errorbar(hwf.distance_A, hwf.hw_zne_linear, yerr=hwf.hw_zne_linear_std, fmt="s", c="#1f77b4", capsize=3, label="Hardware + ZNE")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)")
ax.set_title("HeH$^+$ potential energy surface, STO-3G: from exact to noisy"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "headline_pes.png", dpi=150)

### Limitations and scaling
- **Fixed parameters.** We optimised on a noiseless simulator and only *evaluated* on noisy devices. A full VQE loop on hardware would need hundreds of jobs, and the optimiser (SPSA would be the realistic choice) would chase shot noise.
- **Shot noise is the hardware bottleneck.** Even at 32768 shots one energy wobbles by several mHa, more than chemical accuracy, and ZNE makes the error bar 1.2× (linear) to 2.3× (quadratic) larger.
- **ZNE assumptions.** It assumes the energy changes smoothly and predictably as we fold gates. Folding only amplifies two-qubit gate noise, not readout or idle noise, so those leave a residual. It also costs 3× the circuits.
- **Tapering makes this system tiny.** 2 qubits and 2 CZ gates for HeH⁺/STO-3G. Bigger bases or molecules grow fast (Section 4: 6-31G already needs 6 qubits after tapering), and the two-qubit gate count of UCCSD grows much faster than the qubit count.
- **Calibration drifts.** The noise-aware layout uses calibration data that changes during the day; the qubits we picked may not stay the best.

# 9. Conclusions
- **Owner:** both
- **Scores on:** Presentation
- **Timebox:** 1:45–2:00

*TODO: 4–6 bullets with numbers. Then 2–3 bullets on what we'd do with more time.*

# Appendix
- **Hardware:** backend *TODO*, job ID(s) *TODO*, date *TODO*
- **Environment:** see the cell below.
- **Credits:** builds on B. Tirado, *Ground-State Energy of a Molecule with VQE*, BasQ Qiskit Fall Fest 2026 tutorial (CC BY 4.0 / Apache 2.0).

In [ ]:
from importlib.metadata import version
import sys
print("python", sys.version.split()[0])
for p in ["qiskit", "qiskit-nature", "qiskit-algorithms", "qiskit-aer", "qiskit-ibm-runtime", "pyscf"]:
    print(f"{p:20s}{version(p)}")